# Week 3 — Part B: Lie Factor, in Code

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

Audit five published graphics using `vizlib.lie_factor()`. No rulers — compute everything.

**Tufte's Lie Factor** = (effect shown in the graphic) / (effect in the data), where effect = (final − initial) / initial.  
Honest = 1.0. Accepted range: 0.95 – 1.05.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from vizlib import lie_factor, lie_verdict, PALETTE, style_defaults

style_defaults()
DATA = os.path.join('..', 'data')
print('vizlib loaded.')

## Question 1 — Lie Factor Table for Graphics A–E

In [ ]:
lf_data = pd.read_csv(os.path.join(DATA, 'case07_lie_factor.csv'))
print(lf_data.to_string(index=False))

In [ ]:
results = []
for _, row in lf_data.iterrows():
    lf, data_eff, graphic_eff = lie_factor(
        row['value_start'], row['value_end'],
        row['ink_start'],   row['ink_end'],
        ink_dimension=int(row['ink_dimension'])
    )
    results.append({
        'chart': row['chart_id'],
        'subject': row['subject'],
        'data_effect':    round(data_eff,   3),
        'graphic_effect': round(graphic_eff, 3),
        'lie_factor':     round(lf, 2),
        'verdict':        lie_verdict(lf),
        'ink_dim':        int(row['ink_dimension'])
    })

results_df = pd.DataFrame(results)
print(results_df[['chart','subject','data_effect','graphic_effect','lie_factor','verdict']].to_string(index=False))

In [ ]:
C = PALETTE['series']
fig, ax = plt.subplots(figsize=(9, 5))
colors = [C[1] if row['lie_factor'] > 1.05 else
          (C[2] if 0.95 <= row['lie_factor'] <= 1.05 else PALETTE['muted'])
          for _, row in results_df.iterrows()]

bars = ax.bar(results_df['chart'], results_df['lie_factor'], color=colors, alpha=0.85, edgecolor='white')
ax.axhline(1.0, color='#888888', lw=1.5, linestyle='--', label='Honest (LF=1)')
ax.axhspan(0.95, 1.05, alpha=0.08, color='green')
for bar, val in zip(bars, results_df['lie_factor']):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.2, f'{val:.1f}', ha='center', fontsize=9)

ax.set_title('Lie Factors for five published graphics — only C is honest (LF=1.0)', loc='left')
ax.set_xlabel('Graphic ID')
ax.set_ylabel('Lie Factor')
ax.set_ylim(0, 80)
ax.legend(frameon=False)
for sp in ('top','right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('partB_lie_factors.png', dpi=150, bbox_inches='tight')
plt.show()

## Question 2 — Which Is Honest, and What Do the Others Do Wrong?

In [ ]:
for _, row in results_df.iterrows():
    print(f"Graphic {row['chart']}: LF = {row['lie_factor']:.2f} — {row['verdict']}")
    print(f"  {lf_data[lf_data['chart_id']==row['chart']]['how_the_graphic_encodes_it'].values[0]}")
    print()

**Graphic C is the only honest one** (LF = 1.00). It draws bars from zero at the correct proportional lengths (5 cm to 6 cm for a 20% data rise → 20% ink rise).

**What the others do wrong:**

- **A (LF ≈ 75):** Bars drawn from a 29.6% baseline instead of zero. A 2-percentage-point rise (30→32%) appears as a 6× rise in ink (4mm → 24mm). The most extreme distortion in the set.
- **B (LF = 2.5):** A coin icon scaled 1.5× in both height *and* width to represent a 50% (1.5×) budget rise. Area grows as the square of the linear scale factor (1.5² = 2.25×), so the visual effect is 2.25× − 1 = 125% vs the data's 50% rise. `ink_dimension=2` is required here.
- **D (LF = 0.25):** Bars drawn 8 cm and 9 cm for a data rise of 100 to 150 (50%). The ink rise is only 12.5% (8→9 cm). The chart *understates* the change — LF < 1.
- **E (LF = 5.0):** Bubble radius set proportional to value (radius 1→4 for value 2→8). Area grows as radius² (1→16), so the apparent effect is 15× vs the data's 3× rise. `ink_dimension=2` required.

## Question 3 — Which Need `ink_dimension=2`?

In [ ]:
print('ink_dimension from the data file:')
for _, row in lf_data.iterrows():
    print(f"  Graphic {row['chart_id']}: ink_dimension = {row['ink_dimension']} — {row['how_the_graphic_encodes_it']}")

**Graphics B and E need `ink_dimension=2`** because their quantities are encoded by a 2-D area (a coin scaled in both dimensions; a bubble sized by radius).

**What goes wrong if you leave it at 1:**  
With `ink_dimension=1`, `lie_factor()` computes the ink ratio as `(ink_end / ink_start) − 1`. For Graphic B this gives `(1.5/1.0) − 1 = 0.50`, matching the data effect exactly — yielding a spurious LF of 1.0 and a false 'honest' verdict. The actual graphic area grew by a factor of 1.5² = 2.25, not 1.5. Using dimension=1 on a 2-D encoding always understates the lie, because it treats a quadratic effect as a linear one.

## Question 4 — The Graphic Below 1: Graphic D (LF = 0.25)

**Why an understated chart is still a distortion:**

Tufte's definition is symmetric: LF < 1 means the graphic *understates* the data effect — the visual change is smaller than what the numbers warrant. This is a distortion in the opposite direction. A 50% rise in an air-pollution index that is drawn to look like a 12.5% rise gives the reader false reassurance. The distortion is real even if it is more socially comfortable than exaggeration.

**Who benefits:**  
The party responsible for the air-pollution index (in context, a government environmental agency or an industrial operator). An understated chart makes a bad trend look less alarming, reducing pressure for intervention or regulatory action. The beneficiary of minimised alarm is usually the entity that produced or endorsed the measurement.

## Question 5 — Graphic F: The Dual-Axis Chart from Case 7

In [ ]:
dual = pd.read_csv(os.path.join(DATA, 'case07_dual_axis_series.csv'))
print(dual)
print('\nProfit in every month:')
print(dual.assign(profit=dual['revenue_pkr_m'] - dual['cost_pkr_m'])[['month','profit']])

**Can the Lie Factor be computed for Chart F?**

No — and explaining why is the point of this question.

Tufte's formula assumes that **one quantity** is encoded by **one piece of ink** on a single consistent scale. The dual-axis chart encodes revenue on one scale (50–80 PKR m) and cost on another (40–56 PKR m). There is no single effect to measure and no single ink dimension to compare it against. The visual 'crossing' of the lines does not correspond to any data event — it is an artefact of how the two y-axis ranges were chosen. The Lie Factor formula *presupposes* a common baseline and a single encoding channel; the dual-axis chart has neither.

**What the data actually shows:**  
Revenue exceeded cost in every month of the period. The minimum monthly profit was 8 PKR m (January). The company was not running at a loss at any point in H1 2026. The 'loss until May' narrative is entirely a product of the scale manipulation, not of the underlying numbers.